# EDA of the entire ME4OH 1993-2014 datset

In [ ]:
import datetime

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.dates import YearLocator, DateFormatter
from matplotlib.ticker import MultipleLocator

from plotting_functions import plot_ohc_map

### Load data

In [ ]:
# Check variables in data:
# data_path = 'ME4OH_EN411_OFAM3/data/en4.1.1/1993-2014/'
# files = [f for f in glob(os.path.join(data_path, "*.nc"))]
# file2read = nc.Dataset(files[0],'r')
# print(file2read.variables.keys())

In [ ]:
df = pd.read_csv("ME4OH_EN411_OFAM3/WORLD_1993_2014.csv")
# Convert dates to datetime, save year month and year indication columns (aids plotting)
df['Date'] = pd.to_datetime(df['Date'])
df['year_month'] = df['Date'].dt.to_period('M')
df['year'] = df['Date'].dt.to_period('Y')
df

In [ ]:
# Filter by Argo platforms
argo_df = df[df['Project name'].isin(['ARGOR', 'ARGOD', 'ARGOA'])].sort_values('Date').reset_index(drop=True)
argo_df

# EDA

- Amount of data points across time
- Amount of data across space
- Plot OHC vs lat long for a single month?

### Monthly mean OHC

In [ ]:
monthly_means = df.groupby(df['Date'].dt.strftime('%m/%Y'))['OHC'].mean().sort_values()
monthly_std = df.groupby(df['Date'].dt.strftime('%m/%Y'))['OHC'].std().sort_values()

In [ ]:
df_monthly = pd.DataFrame(monthly_means).reset_index()
df_monthly['Date'] = pd.to_datetime(df_monthly['Date'], format="%m/%Y")
df_monthly["std"] = monthly_std
df_monthly = df_monthly.sort_values('Date').reset_index(drop=True)
df_monthly

In [ ]:
# sns.lineplot(df_monthly, x='Date', y='OHC')
plt.plot(df_monthly['Date'], df_monthly['OHC'])
plt.ylabel("Mean OHC across all measurements (per month)")
plt.xlabel("Year")

### Monthly counts

In [ ]:
df['year_month'] = df['Date'].dt.to_period('M')
monthly_counts = df.groupby('year_month').size()

df_counts = pd.DataFrame(monthly_counts).reset_index()
df_counts.rename(columns={0: 'Count', 'year_month': 'Month'}, inplace=True)
df_counts['Month'] = pd.to_datetime(df_counts['Month'].astype(str), format="%Y-%m")
df_counts

In [ ]:
argo_df['year_month'] = argo_df['Date'].dt.to_period('M')
monthly_argo_counts = argo_df.groupby('year_month').size()

df_argo_counts = pd.DataFrame(monthly_argo_counts).reset_index()
df_argo_counts.rename(columns={0: 'Count', 'year_month': 'Month'}, inplace=True)
df_argo_counts['Month'] = pd.to_datetime(df_argo_counts['Month'].astype(str), format="%Y-%m")
df_argo_counts

In [ ]:
month_list = pd.date_range('1993-1-1','2000-1-1', freq='MS').strftime('%m/%Y').tolist()
df_zeros = pd.DataFrame({
    "Month": month_list,
    "Count": np.zeros(len(month_list)),
})
df_zeros['Month'] = pd.to_datetime(df_zeros['Month'], format="%m/%Y")
df_zeros

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(df_counts['Month'], df_counts['Count'], marker='.')
plt.grid(True)
plt.ylabel("Number of observations per month")
plt.xlabel("Year")
plt.axvline(x=datetime.datetime(2000, 1, 1), c='red', ls='--')
ax = plt.gca()
ax.set_xlim([datetime.datetime(1993,1,1), datetime.datetime(2015,1,1)])

ax.xaxis.set_major_locator(YearLocator(1))
ax.xaxis.set_major_formatter(DateFormatter('%Y'))
plt.xticks(rotation=90)

plt.ticklabel_format(axis='y', style='sci', scilimits=(3,3))
plt.savefig("images/monthly_observations.png")
plt.show()
plt.close()


In [ ]:
comp_df = pd.DataFrame()
comp_df['Month'] = pd.date_range(start = '1993-01-01', end = '2014-12-01', freq='MS')
comp_df['Month'] = pd.to_datetime(comp_df['Month'])
comp_df['Argo'] = pd.concat([df_zeros, pd.DataFrame(df_argo_counts)], ignore_index=True)['Count']
comp_df['Other'] = pd.DataFrame(df_counts)['Count'] - pd.concat([df_zeros, pd.DataFrame(df_argo_counts)], ignore_index=True)['Count']
comp_df

In [ ]:
comp_df_plot = comp_df.set_index('Month')

comp_df_plot.index = comp_df_plot.index.to_period('M').year  # Using period indexing from pandas, without this get an error
ax = comp_df_plot.plot(kind='bar', stacked=True, figsize=(12, 6), width=0.6)
ax.set_ylabel('Number of observations per month')
ax.set_xlabel('Year')
ax.set_title('Monthly observation counts: Argo vs Other platforms')

ax.xaxis.set_major_locator(MultipleLocator(12))  # Have stored years only as the period index, so shows only year
# If you want to plot by month, remove .year from comp_df_plot.index.to_period('M').year above and then will show months and years
plt.ticklabel_format(axis='y', style='sci', scilimits=(3,3))

plt.tight_layout()
plt.grid()
plt.legend(title="Platform type")
plt.savefig("images/monthly_observations_argo_comparison.png")
plt.show()
plt.close()

In [ ]:
df["Platform number"].unique()

In [ ]:
df["Project name"].unique()

In [ ]:
# df.plot.scatter(x='Date', y='OHC', c='Platform number')

In [ ]:
platform_counts = df.groupby(['Project name', 'Platform number']).size()

df_counts = pd.DataFrame(platform_counts).reset_index()
df_counts.rename(columns={0: 'Count'}, inplace=True)
df_counts[df_counts['Platform number'].duplicated(keep=False)].to_csv('duplcated_platforms.csv')


In [ ]:
df['Unique ID'] = df['Project name'] + ' ' + df['Platform number']

## Plot OHC in latitude and longitude
Plot for each month

In [ ]:
month_list = pd.date_range('1993-1-1','2014-12-31', freq='MS').strftime('%Y-%m').tolist()
num_plots = 10
plot_months = [month_list[i] for i in range(0, len(month_list), len(month_list)//num_plots)]

In [ ]:
for n,g in df.groupby('year_month'):
    if str(n) in plot_months: 
        plot_ohc_map(g, n, min(df.OHC), max(df.OHC))

In [ ]:
df['year'] = df['Date'].dt.to_period('Y')

In [ ]:
for n,g in df.groupby('year'):
    plot_ohc_map(g, n, min(df.OHC), max(df.OHC))